# Notebook Cell Index and Descriptions
!!caution!! - The parameters used here are very high for better resolution of gifs and images. Change those before running for testing purpose. the current parameters will take each cell to run for 1 hour or more even in current high end systems.(year - 2025)

**Cell 1 (Markdown):**
- This cell provides an index and description of each code cell in the notebook.
- The library that is used in this notebook is Meep(MIT Electromagnetic Equation Propagation) which is an open-source software package for simulating electromagnetic systems using the finite-difference time-domain (FDTD) method. os, io, numpy, matplotlib, PIL are other libraries used for file handling, numerical operations, plotting and image processing respectively.

**Cell 2 (Code):**
- Outputs: Gifs of Optical band structures and Hz fields for different geometry offsets(distance of holes from center).
- varying parameters in the Gifs are the geometry offset values only all other remain constant.

**cell 3 (Code):**
- Outputs: Gifs of Optical band structures transition and Hz fields transition for different geometry offsets(distance of holes from center).
- Transitions region are just narrow variation of geometry offset values near phase transition point.(found from previous cell).







In [ ]:
# Cell 2: Generate GIFs of band structure and Hz field for varying geometry offsets

import meep as mp
from meep import mpb
import numpy as np
import matplotlib.pyplot as plt
import os
from PIL import Image
import io

# -------------------------------------------------
# Create output directory for GIFs
# -------------------------------------------------
output_dir = "gif"
if not os.path.exists(output_dir):
    os.makedirs(output_dir)
    print(f"Created directory: {output_dir}")

# -------------------------------------------------
# Simulation Parameters
# -------------------------------------------------
# Physical lattice constant (nm) for THz conversion (MPB uses a=1)
a_nm = 45900.0  # lattice constant in nanometers (um)
a = 1.0         # normalized lattice constant
r = 0.2 * a     # hole radius
eps_bg = 3.42**2  # background permittivity (silicon)
resolution = 1000 # grid resolution
num_bands = 6   # number of bands to compute

# Speed of light and conversion factor from normalized freq to THz
c0 = 299_792_458.0  # speed of light in m/s
freq_scale_THz = (c0 / (a_nm * 1e-9)) / 1e12  # THz per normalized frequency unit

# -------------------------------------------------
# Define k-point path for band structure calculation
# -------------------------------------------------
Gamma = mp.Vector3(0, 0)
X = mp.Vector3(0.5, 0)
M = mp.Vector3(0.5, 0.5)
k_points_base = mp.interpolate(20, [Gamma, X, M, Gamma])

# -------------------------------------------------
# Create photonic crystal lattice
# -------------------------------------------------
lattice = mp.Lattice(size=mp.Vector3(1, 1))

# -------------------------------------------------
# Offset range for geometry (distance of holes from center)
# -------------------------------------------------
offset_values = np.linspace(0.10, 0.4, 100)

band_frames = []  # store band structure images for GIF
hz_frames = []    # store Hz field images for GIF

print(f"Generating GIFs with offsets: {offset_values}")
print(f"Resolution: {resolution}")
print(f"Using lattice constant a = {a_nm} nm; freq scale = {freq_scale_THz:.3f} THz per unit")

for idx, offset in enumerate(offset_values):
    print(f"\nProcessing frame {idx+1}/{len(offset_values)}: offset = {offset:.2f}*a")
    
    offset_a = offset * a
    
    # -------------------------------------------------
    # Create geometry for current offset
    # -------------------------------------------------
    geometry = [
        mp.Cylinder(
            radius=r,
            height=mp.inf,
            center=mp.Vector3(+offset_a, +offset_a),
            material=mp.Medium(epsilon=1.0)
        ),
        mp.Cylinder(
            radius=r,
            height=mp.inf,
            center=mp.Vector3(+offset_a, -offset_a),
            material=mp.Medium(epsilon=1.0)
        ),
        mp.Cylinder(
            radius=r,
            height=mp.inf,
            center=mp.Vector3(-offset_a, +offset_a),
            material=mp.Medium(epsilon=1.0)
        ),
        mp.Cylinder(
            radius=r,
            height=mp.inf,
            center=mp.Vector3(-offset_a, -offset_a),
            material=mp.Medium(epsilon=1.0)
        ),
    ]
    
    # -------------------------------------------------
    # MPB solver for band structure
    # -------------------------------------------------
    ms = mpb.ModeSolver(
        geometry_lattice=lattice,
        geometry=geometry,
        k_points=k_points_base,
        resolution=resolution,
        num_bands=num_bands,
        default_material=mp.Medium(epsilon=eps_bg)
    )
    
    ms.run_te()  # Run TE mode calculation
    bands = np.array(ms.all_freqs)
    bands_THz = bands * freq_scale_THz  # Convert to THz
    
    # -------------------------------------------------
    # Calculate minimum adjacent band gap (THz)
    # -------------------------------------------------
    gaps = np.diff(bands_THz, axis=1)
    min_gap = gaps.min()
    gap_loc = np.unravel_index(np.argmin(gaps), gaps.shape)
    print(f"Min adjacent gap: {min_gap:.3f} THz (between bands {gap_loc[1]+1} and {gap_loc[1]+2} along k-point {gap_loc[0]})")
    
    # -------------------------------------------------
    # Plot 1: Band Structure (THz, colored bands, gap annotation)
    # -------------------------------------------------
    fig, ax = plt.subplots(figsize=(6, 5))
    cmap = plt.get_cmap('tab10')
    colors = [cmap(i % 10) for i in range(num_bands)]
    for n in range(num_bands):
        ax.plot(bands_THz[:, n], color=colors[n], linewidth=1.5, label=f'Band {n+1}')
    # Compute gap between band 1 and band 2 (indices 0 and 1)
    max_band1 = bands_THz[:, 0].max()
    min_band2 = bands_THz[:, 1].min()
    gap_thz = min_band2 - max_band1
    # Draw dotted horizontal lines at max(band1) and min(band2)
    x_min = 0
    x_max = bands_THz.shape[0] - 1
    ax.hlines(max_band1, x_min, x_max, colors='gray', linestyles=':', linewidth=1)
    ax.hlines(min_band2, x_min, x_max, colors='gray', linestyles=':', linewidth=1)
    # Annotate the gap value (THz) between the two lines
    mid_x = 0.5 * (x_min + x_max)
    mid_y = max_band1 + 0.5 * max(0.0, gap_thz)
    ax.text(mid_x, mid_y, f"Gap = {gap_thz:.3f} THz", color='black',
            fontsize=10, ha='center', va='center', bbox=dict(facecolor='white', alpha=0.7, edgecolor='none'))
    xticks = [0, 20, 40, 60]
    xlabels = [r"$\Gamma$", r"$X$", r"$M$", r"$\Gamma$"]
    ax.set_xticks(xticks)
    ax.set_xticklabels(xlabels)
    ax.set_ylabel("Frequency (THz)", fontsize=12)
    ax.set_title(f"Photonic Band Structure (offset = {offset:.2f}a)", fontsize=13)
    ax.grid(True, alpha=0.3)
    ax.legend(loc='upper right', fontsize=8)
    # Convert to image for GIF
    buf = io.BytesIO()
    fig.savefig(buf, format='png', dpi=100, bbox_inches='tight')
    buf.seek(0)
    band_frames.append(Image.open(buf).copy())
    plt.close(fig)
    
    # -------------------------------------------------
    # Plot 2: Hz Field at M point
    # -------------------------------------------------
    ms.k_points = [M]
    ms.run_te()
    
    eps = ms.get_epsilon()
    Hz = ms.get_hfield(1, bloch_phase=True)
    Hz = np.real(Hz[:, :, 0, 2])
    
    # Create Hz field plot
    fig, ax = plt.subplots(figsize=(6, 6))
    im = ax.imshow(
        Hz.T, origin='lower',
        extent=[-0.5, 0.5, -0.5, 0.5],
        cmap='RdBu'
    )
    ax.set_title(f"Hz Field at M-point (offset = {offset:.2f}a)", fontsize=13)
    ax.set_xlabel("x / a", fontsize=11)
    ax.set_ylabel("y / a", fontsize=11)
    plt.colorbar(im, ax=ax, label="Hz")
    # Convert to image for GIF
    buf = io.BytesIO()
    fig.savefig(buf, format='png', dpi=100, bbox_inches='tight')
    buf.seek(0)
    hz_frames.append(Image.open(buf).copy())
    plt.close(fig)

# -------------------------------------------------
# Create GIFs from collected frames
# -------------------------------------------------
print("\nCreating GIFs...")

# Save band structure GIF
band_gif_path = os.path.join(output_dir, "bandstructure.gif")
band_frames[0].save(
    band_gif_path,
    save_all=True,
    append_images=band_frames[1:],
    duration=500,
    loop=0
)
print(f"Saved: {band_gif_path}")

# Save Hz field GIF
hz_gif_path = os.path.join(output_dir, "hz_field.gif")
hz_frames[0].save(
    hz_gif_path,
    save_all=True,
    append_images=hz_frames[1:],
    duration=500,
    loop=0
)
print(f"Saved: {hz_gif_path}")

print("\nGIF generation complete!")
print(f"Band structure GIF: {band_gif_path}")
print(f"Hz field GIF: {hz_gif_path}")


Generating GIFs with offsets: [0.1   0.175 0.25  0.325 0.4  ]
Resolution: 10
Using lattice constant a = 45900.0 nm; freq scale = 6.531 THz per unit

Processing frame 1/5: offset = 0.10*a
Initializing eigensolver data
Computing 4 bands with 1e-07 tolerance
Working in 2 dimensions.
Grid size is 10 x 10 x 1.
Solving for 4 bands at a time.
Creating Maxwell data...
Mesh size is 3.
Lattice vectors:
     (1, 0, 0)
     (0, 1, 0)
     (0, 0, 1)
Cell volume = 1
Reciprocal lattice vectors (/ 2 pi):
     (1, -0, 0)
     (-0, 1, -0)
     (0, -0, 1)
Geometric objects:
     cylinder, center = (0.1,0.1,0)
          radius 0.2, height 1e+20, axis (0, 0, 1)
     cylinder, center = (0.1,-0.1,0)
          radius 0.2, height 1e+20, axis (0, 0, 1)
     cylinder, center = (-0.1,0.1,0)
          radius 0.2, height 1e+20, axis (0, 0, 1)
     cylinder, center = (-0.1,-0.1,0)
          radius 0.2, height 1e+20, axis (0, 0, 1)
Geometric object tree has depth 1 and 4 object nodes (vs. 4 actual objects)
Initializi

In [ ]:
import meep as mp
from meep import mpb
import numpy as np
import matplotlib.pyplot as plt
import os
from PIL import Image
import io

# -------------------------------------------------
# Create output directory
# -------------------------------------------------
output_dir = "gif"
if not os.path.exists(output_dir):
    os.makedirs(output_dir)
    print(f"Created directory: {output_dir}")

# -------------------------------------------------
# Parameters
# -------------------------------------------------
# Lattice constant in nanometers (used only for unit conversion; MPB still uses a=1)
a_nm = 45900.0
a = 1.0
r = 0.2 * a  # hole radius
eps_bg = 3.42**2  # background permittivity
resolution = 1000
num_bands = 6

# Speed of light and conversion factor from normalized freq (\omega a / 2\pi c) to THz
c0 = 299_792_458.0  # m/s
freq_scale_THz = (c0 / (a_nm * 1e-9)) / 1e12  # THz per normalized frequency unit

# k-point path
Gamma = mp.Vector3(0, 0)
X = mp.Vector3(0.5, 0)
M = mp.Vector3(0.5, 0.5)
k_points_base = mp.interpolate(20, [Gamma, X, M, Gamma])

# Create lattice
lattice = mp.Lattice(size=mp.Vector3(1, 1))

# -------------------------------------------------
# Offset range from 0.23*a to 0.27*a
# -------------------------------------------------
offset_values = np.linspace(0.23, 0.27, 40)

band_frames = []
hz_frames = []

print(f"Generating GIFs with offsets: {offset_values}")
print(f"Resolution: {resolution}")
print(f"Using lattice constant a = {a_nm} nm; freq scale = {freq_scale_THz:.3f} THz per unit")

for idx, offset in enumerate(offset_values):
    print(f"\nProcessing frame {idx+1}/{len(offset_values)}: offset = {offset:.2f}*a")
    
    offset_a = offset * a
    
    # Create geometry with current offset
    geometry = [
        mp.Cylinder(
            radius=r,
            height=mp.inf,
            center=mp.Vector3(+offset_a, +offset_a),
            material=mp.Medium(epsilon=1.0)
        ),
        mp.Cylinder(
            radius=r,
            height=mp.inf,
            center=mp.Vector3(+offset_a, -offset_a),
            material=mp.Medium(epsilon=1.0)
        ),
        mp.Cylinder(
            radius=r,
            height=mp.inf,
            center=mp.Vector3(-offset_a, +offset_a),
            material=mp.Medium(epsilon=1.0)
        ),
        mp.Cylinder(
            radius=r,
            height=mp.inf,
            center=mp.Vector3(-offset_a, -offset_a),
            material=mp.Medium(epsilon=1.0)
        ),
    ]
    
    # -------------------------------------------------
    # MPB solver for band structure
    # -------------------------------------------------
    ms = mpb.ModeSolver(
        geometry_lattice=lattice,
        geometry=geometry,
        k_points=k_points_base,
        resolution=resolution,
        num_bands=num_bands,
        default_material=mp.Medium(epsilon=eps_bg)
    )
    
    ms.run_te()
    bands = np.array(ms.all_freqs)
    bands_THz = bands * freq_scale_THz
    
    # Minimum adjacent band gap across the path (THz)
    gaps = np.diff(bands_THz, axis=1)
    min_gap = gaps.min()
    gap_loc = np.unravel_index(np.argmin(gaps), gaps.shape)
    print(f"Min adjacent gap: {min_gap:.3f} THz (between bands {gap_loc[1]+1} and {gap_loc[1]+2} along k-point {gap_loc[0]})")
    
    # -------------------------------------------------
    # Plot 1: Band Structure (colored bands + gap annotation)
    # -------------------------------------------------
    fig, ax = plt.subplots(figsize=(6, 5))
    # Colors for bands
    cmap = plt.get_cmap('tab10')
    colors = [cmap(i % 10) for i in range(num_bands)]
    # Plot each band with a different color
    for n in range(num_bands):
        ax.plot(bands_THz[:, n], color=colors[n], linewidth=1.5, label=f'Band {n+1}')
    # Compute gap between band 1 and band 2 (indices 0 and 1)
    max_band1 = bands_THz[:, 0].max()
    min_band2 = bands_THz[:, 1].min()
    gap_thz = min_band2 - max_band1
    # Draw dotted horizontal lines at max(band1) and min(band2)
    x_min = 0
    x_max = bands_THz.shape[0] - 1
    ax.hlines(max_band1, x_min, x_max, colors='gray', linestyles=':', linewidth=1)
    ax.hlines(min_band2, x_min, x_max, colors='gray', linestyles=':', linewidth=1)
    # Annotate the gap value (THz) between the two lines
    mid_x = 0.5 * (x_min + x_max)
    mid_y = max_band1 + 0.5 * max(0.0, gap_thz)
    ax.text(mid_x, mid_y, f"Gap = {gap_thz:.3f} THz", color='black',
            fontsize=10, ha='center', va='center', bbox=dict(facecolor='white', alpha=0.7, edgecolor='none'))
    xticks = [0, 20, 40, 60]
    xlabels = [r"$\Gamma$", r"$X$", r"$M$", r"$\Gamma$"]
    ax.set_xticks(xticks)
    ax.set_xticklabels(xlabels)
    ax.set_ylabel("Frequency (THz)", fontsize=12)
    ax.set_title(f"Photonic Band Structure (offset = {offset:.2f}a)", fontsize=13)
    ax.grid(True, alpha=0.3)
    ax.legend(loc='upper right', fontsize=8)
    # Convert to image
    buf = io.BytesIO()
    fig.savefig(buf, format='png', dpi=100, bbox_inches='tight')
    buf.seek(0)
    band_frames.append(Image.open(buf).copy())
    plt.close(fig)
    
    # -------------------------------------------------
    # Plot 2: Hz Field at M point
    # -------------------------------------------------
    ms.k_points = [M]
    ms.run_te()
    
    eps = ms.get_epsilon()
    Hz = ms.get_hfield(1, bloch_phase=True)
    Hz = np.real(Hz[:, :, 0, 2])
    
    # Create Hz field plot
    fig, ax = plt.subplots(figsize=(6, 6))
    im = ax.imshow(
        Hz.T, origin='lower',
        extent=[-0.5, 0.5, -0.5, 0.5],
        cmap='RdBu'
    )
    ax.set_title(f"Hz Field at M-point (offset = {offset:.2f}a)", fontsize=13)
    ax.set_xlabel("x / a", fontsize=11)
    ax.set_ylabel("y / a", fontsize=11)
    plt.colorbar(im, ax=ax, label="Hz")
    # Convert to image
    buf = io.BytesIO()
    fig.savefig(buf, format='png', dpi=100, bbox_inches='tight')
    buf.seek(0)
    hz_frames.append(Image.open(buf).copy())
    plt.close(fig)

# -------------------------------------------------
# Create GIFs
# -------------------------------------------------
print("\nCreating GIFs...")

# Save band structure GIF
band_gif_path = os.path.join(output_dir, "bandstructure_transition.gif")
band_frames[0].save(
    band_gif_path,
    save_all=True,
    append_images=band_frames[1:],
    duration=500,
    loop=0
)
print(f"Saved: {band_gif_path}")

# Save Hz field GIF
hz_gif_path = os.path.join(output_dir, "hz_field_transition.gif")
hz_frames[0].save(
    hz_gif_path,
    save_all=True,
    append_images=hz_frames[1:],
    duration=500,
    loop=0
)
print(f"Saved: {hz_gif_path}")

print("\nGIF generation complete!")
print(f"Band structure GIF: {band_gif_path}")
print(f"Hz field GIF: {hz_gif_path}")

In [ ]:
# Generate GIFs for varying hole radius (r)
import meep as mp
from meep import mpb
import numpy as np
import matplotlib.pyplot as plt
import os
from PIL import Image
import io

# Output directory for GIFs
output_dir = "gif"
if not os.path.exists(output_dir):
    os.makedirs(output_dir)
    print(f"Created directory: {output_dir}")

# Simulation parameters
a_nm = 45900.0  # lattice constant in nanometers
a = 1.0         # normalized lattice constant
eps_bg = 3.42**2  # background permittivity (silicon)
resolution = 1000 # grid resolution
num_bands = 6
offset = 0.35 * a

# Speed of light and conversion factor from normalized freq to THz
c0 = 299_792_458.0  # m/s
freq_scale_THz = (c0 / (a_nm * 1e-9)) / 1e12  # THz per normalized frequency unit

# k-point path
Gamma = mp.Vector3(0, 0)
X = mp.Vector3(0.5, 0)
M = mp.Vector3(0.5, 0.5)
k_points_base = mp.interpolate(20, [Gamma, X, M, Gamma])

# Lattice
lattice = mp.Lattice(size=mp.Vector3(1, 1))

# Range of radii to sweep (as a fraction of a)
radius_values = np.linspace(0.00, 0.3, 60)

band_frames = []
hz_frames = []

print(f"Generating GIFs with radii: {radius_values}")
print(f"Resolution: {resolution}")
print(f"Using lattice constant a = {a_nm} nm; freq scale = {freq_scale_THz:.3f} THz per unit")

for idx, r_frac in enumerate(radius_values):
    print(f"\nProcessing frame {idx+1}/{len(radius_values)}: radius = {r_frac:.2f}*a")
    r = r_frac * a

    # Geometry with current radius, fixed offset (e.g., 0.2*a)
    geometry = [
        mp.Cylinder(radius=r, height=mp.inf, center=mp.Vector3(+offset, +offset), material=mp.Medium(epsilon=1.0)),
        mp.Cylinder(radius=r, height=mp.inf, center=mp.Vector3(+offset, -offset), material=mp.Medium(epsilon=1.0)),
        mp.Cylinder(radius=r, height=mp.inf, center=mp.Vector3(-offset, +offset), material=mp.Medium(epsilon=1.0)),
        mp.Cylinder(radius=r, height=mp.inf, center=mp.Vector3(-offset, -offset), material=mp.Medium(epsilon=1.0)),
    ]

    # MPB solver for band structure
    ms = mpb.ModeSolver(
        geometry_lattice=lattice,
        geometry=geometry,
        k_points=k_points_base,
        resolution=resolution,
        num_bands=num_bands,
        default_material=mp.Medium(epsilon=eps_bg)
    )

    ms.run_te()
    bands = np.array(ms.all_freqs)
    bands_THz = bands * freq_scale_THz

    # Plot 1: Band Structure (THz, colored bands, gap annotation)
    fig, ax = plt.subplots(figsize=(6, 5))
    cmap = plt.get_cmap('tab10')
    colors = [cmap(i % 10) for i in range(num_bands)]
    for n in range(num_bands):
        ax.plot(bands_THz[:, n], color=colors[n], linewidth=1.5, label=f'Band {n+1}')
    max_band1 = bands_THz[:, 0].max()
    min_band2 = bands_THz[:, 1].min()
    gap_thz = min_band2 - max_band1
    x_min = 0
    x_max = bands_THz.shape[0] - 1
    ax.hlines(max_band1, x_min, x_max, colors='gray', linestyles=':', linewidth=1)
    ax.hlines(min_band2, x_min, x_max, colors='gray', linestyles=':', linewidth=1)
    mid_x = 0.5 * (x_min + x_max)
    mid_y = max_band1 + 0.5 * max(0.0, gap_thz)
    ax.text(mid_x, mid_y, f"Gap = {gap_thz:.3f} THz", color='black',
            fontsize=10, ha='center', va='center', bbox=dict(facecolor='white', alpha=0.7, edgecolor='none'))
    xticks = [0, 20, 40, 60]
    xlabels = [r"$\Gamma$", r"$X$", r"$M$", r"$\Gamma$"]
    ax.set_xticks(xticks)
    ax.set_xticklabels(xlabels)
    ax.set_ylabel("Frequency (THz)", fontsize=12)
    ax.set_title(f"Band Structure (radius = {r_frac:.2f}a)", fontsize=13)
    ax.grid(True, alpha=0.3)
    ax.legend(loc='upper right', fontsize=8)
    buf = io.BytesIO()
    fig.savefig(buf, format='png', dpi=100, bbox_inches='tight')
    buf.seek(0)
    band_frames.append(Image.open(buf).copy())
    plt.close(fig)

    # Plot 2: Hz Field at M point
    ms.k_points = [M]
    ms.run_te()
    Hz = np.real(ms.get_hfield(1, bloch_phase=True)[:, :, 0, 2])
    fig, ax = plt.subplots(figsize=(6, 6))
    im = ax.imshow(
        Hz.T, origin='lower',
        extent=[-0.5, 0.5, -0.5, 0.5],
        cmap='RdBu'
    )
    ax.set_title(f"Hz Field at M-point (radius = {r_frac:.2f}a)", fontsize=13)
    ax.set_xlabel("x / a", fontsize=11)
    ax.set_ylabel("y / a", fontsize=11)
    plt.colorbar(im, ax=ax, label="Hz")
    buf = io.BytesIO()
    fig.savefig(buf, format='png', dpi=100, bbox_inches='tight')
    buf.seek(0)
    hz_frames.append(Image.open(buf).copy())
    plt.close(fig)

# Create GIFs
print("\nCreating GIFs...")

band_gif_path = os.path.join(output_dir, "bandstructure_radius.gif")
band_frames[0].save(
    band_gif_path,
    save_all=True,
    append_images=band_frames[1:],
    duration=500,
    loop=0
)
print(f"Saved: {band_gif_path}")

hz_gif_path = os.path.join(output_dir, "hz_field_radius.gif")
hz_frames[0].save(
    hz_gif_path,
    save_all=True,
    append_images=hz_frames[1:],
    duration=500,
    loop=0
)
print(f"Saved: {hz_gif_path}")

print("\nGIF generation complete!")
print(f"Band structure GIF: {band_gif_path}")
print(f"Hz field GIF: {hz_gif_path}")

Generating GIFs with radii: [0.1  0.25 0.4 ]
Resolution: 10
Using lattice constant a = 45900.0 nm; freq scale = 6.531 THz per unit

Processing frame 1/3: radius = 0.10*a
Initializing eigensolver data
Computing 6 bands with 1e-07 tolerance
Working in 2 dimensions.
Grid size is 10 x 10 x 1.
Solving for 6 bands at a time.
Creating Maxwell data...
Mesh size is 3.
Lattice vectors:
     (1, 0, 0)
     (0, 1, 0)
     (0, 0, 1)
Cell volume = 1
Reciprocal lattice vectors (/ 2 pi):
     (1, -0, 0)
     (-0, 1, -0)
     (0, -0, 1)
Geometric objects:
     cylinder, center = (0.25,0.25,0)
          radius 0.1, height 1e+20, axis (0, 0, 1)
     cylinder, center = (0.25,-0.25,0)
          radius 0.1, height 1e+20, axis (0, 0, 1)
     cylinder, center = (-0.25,0.25,0)
          radius 0.1, height 1e+20, axis (0, 0, 1)
     cylinder, center = (-0.25,-0.25,0)
          radius 0.1, height 1e+20, axis (0, 0, 1)
Geometric object tree has depth 2 and 4 object nodes (vs. 4 actual objects)
Initializing epsilo